# B1 — Uber Fare Prediction: Linear vs Random Forest, outliers, correlation, R2, RMSE

**Laboratory Practice III (410246), Group B**

Dataset reference: https://www.kaggle.com/yasserh/uber-fares-dataset

Run cells top-to-bottom. Set `CSV_PATH` to your downloaded CSV file to use real data; `None` runs the included synthetic demonstration. Results on synthetic data are not real-world model performance.

In [ ]:
# Set to a local CSV filename/path after downloading the Kaggle dataset.
# Keep None to run with built-in synthetic demonstration data.
CSV_PATH = None  # Example: 'uber.csv'

## Implementation and evaluation

In [ ]:
pip install pandas numpy matplotlib seaborn scikit-learn


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Load dataset
df = pd.read_csv("uber.csv")

print("First five records:")
print(df.head())

print("\nMissing values:")
print(df.isnull().sum())

# Remove missing and invalid values
df = df.dropna()
df = df[df["fare_amount"] > 0]
df = df[df["passenger_count"] > 0]

# Calculate distance using Haversine formula
lat1 = np.radians(df["pickup_latitude"])
lon1 = np.radians(df["pickup_longitude"])
lat2 = np.radians(df["dropoff_latitude"])
lon2 = np.radians(df["dropoff_longitude"])

a = (np.sin((lat2-lat1)/2)**2 +
     np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2)

df["distance"] = 6371 * 2 * np.arcsin(np.sqrt(a))

# Extract pickup hour
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["pickup_hour"] = df["pickup_datetime"].dt.hour

# Remove unnecessary columns
df = df[["distance", "passenger_count", "pickup_hour", "fare_amount"]]

# Remove zero-distance rides
df = df[df["distance"] > 0]

# Detect and remove fare outliers using IQR
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("\nOutliers:", len(df[(df["fare_amount"] < lower) |
                            (df["fare_amount"] > upper)]))

df = df[(df["fare_amount"] >= lower) &
        (df["fare_amount"] <= upper)]

# Check correlation
print("\nCorrelation:")
print(df.corr())

sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# Split features and target
X = df[["distance", "passenger_count", "pickup_hour"]]
y = df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
pred1 = lr.predict(X_test)

# Random Forest Regression
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
pred2 = rf.predict(X_test)

# Evaluate models
print("\nLinear Regression")
print("R2 Score:", r2_score(y_test, pred1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred1)))

print("\nRandom Forest Regression")
print("R2 Score:", r2_score(y_test, pred2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred2)))
